# Janci Kumari
# Track: Data Science
# Task 5: Sales Prediction Using Python

Oasis Infobyte Internship (OIBSIP)

## Objective
Build regression models that predict product sales from advertising spend on TV, Radio and Newspaper.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Classic Advertising dataset (TV, Radio, Newspaper spend -> Sales)
url = "https://www.statlearning.com/s/Advertising.csv"
df = pd.read_csv(url)

# Drop the unnamed index column if the file has one
df = df.loc[:, ~df.columns.str.contains('^Unnamed|^X$')]

# Use consistent column names
df.columns = ['TV', 'Radio', 'Newspaper', 'Sales']
df.head()

In [ ]:
# Basic checks: shape, nulls, data types, descriptive statistics
print("Shape:", df.shape)
print("\nNull values:\n", df.isnull().sum())
print("\nData types:\n", df.dtypes)
df.describe()

## Exploratory Data Analysis

- The dataset has 200 rows and 4 columns: TV, Radio and Newspaper advertising spend, and Sales.
- All columns are numeric (float) and there are no missing values, so no cleaning was needed.
- TV has the widest range of spend (mean about 147), followed by Newspaper and Radio. Sales has a mean of about 14.

In [ ]:
# Pairplot of all features
sns.pairplot(df)
plt.show()

# Individual scatter plots: Sales vs each advertising channel
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['TV', 'Radio', 'Newspaper']):
    sns.regplot(x=col, y='Sales', data=df, ax=ax)
    ax.set_title(f"Sales vs {col} spend")
plt.tight_layout()
plt.show()

# Correlation matrix heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Correlation matrix")
plt.show()

## Observations from the Scatter Plots

- Sales vs TV: a clear, strong upward trend. Sales increase as TV spend increases. The points spread out more at high TV spend.
- Sales vs Radio: a positive trend, but with more scatter than TV, so the relationship is weaker.
- Sales vs Newspaper: the points are widely scattered and the trend line is flat, so there is only a weak relationship.

## Correlation Analysis

The correlation of each channel with Sales is: TV 0.78, Radio 0.58 and Newspaper 0.23. TV has the strongest linear relationship with Sales, followed by Radio, while Newspaper is weak.

The three advertising channels are only weakly correlated with each other (TV-Radio 0.05, TV-Newspaper 0.06, Radio-Newspaper 0.35), so multicollinearity is not a serious problem for the regression models.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Features and target
X = df[['TV', 'Radio', 'Newspaper']]
y = df['Sales']

# 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Linear Regression is the baseline, Random Forest is the second model
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42)
}

results = []
preds = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    preds[name] = pred
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    cv_r2 = cross_val_score(model, X, y, cv=5, scoring='r2').mean()
    results.append([name, mae, rmse, r2, cv_r2])

res = pd.DataFrame(results, columns=['Model', 'MAE', 'RMSE', 'R2 (test)', 'R2 (5-fold CV)']).round(3)
res

## Model Evaluation

The data was split 80/20 (160 training and 40 test samples). Two regression models were trained and evaluated using MAE, RMSE and R2, plus 5-fold cross-validated R2.

- Linear Regression (baseline): MAE 1.461, RMSE 1.782, R2 0.899 on the test set, cross-validated R2 0.887
- Random Forest Regressor: MAE 0.629, RMSE 0.757, R2 0.982 on the test set, cross-validated R2 0.976

Random Forest is the best model. Its errors are less than half of the Linear Regression errors, and it wins on both the test set and cross-validation, so the difference is not just because of the small test set.

In [ ]:
# Pick the best model by cross-validated R2
best_name = res.sort_values('R2 (5-fold CV)').iloc[-1]['Model']
print("Best model:", best_name)

# Residual plot for the best model
residuals = y_test - preds[best_name]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(preds[best_name], residuals)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel("Predicted Sales")
axes[0].set_ylabel("Residual (Actual - Predicted)")
axes[0].set_title(f"Residuals vs Predicted ({best_name})")
sns.histplot(residuals, kde=True, ax=axes[1])
axes[1].set_xlabel("Residual")
axes[1].set_title("Distribution of residuals")
plt.tight_layout()
plt.show()

# Which channel matters most? Coefficients (Linear Regression) and importance (Random Forest)
coef = pd.Series(models["Linear Regression"].coef_, index=X.columns)
imp = pd.Series(models["Random Forest"].feature_importances_, index=X.columns)
summary = pd.DataFrame({'LR coefficient': coef, 'RF importance': imp}).round(3)
print(summary)
summary.plot(kind='bar', subplots=True, figsize=(8, 4), legend=False)
plt.tight_layout()
plt.show()

## Residual Analysis (Random Forest)

The residuals are scattered around zero with no clear curve or pattern, and most of them lie between -1 and +1. This means the errors look random and not systematic. The histogram of residuals is centred close to zero. A few points at high predicted sales have larger errors (up to about +1.5), so the model is slightly less precise for the highest sales values. The test set has only 40 samples, so this is only an approximate check.

## Interpretation: Which Channel Has the Highest Impact?

- TV has the highest overall impact. It has the highest feature importance in the Random Forest (0.625) and the strongest correlation with Sales (0.78).
- Radio is second overall (importance 0.362), but in Linear Regression it has the largest coefficient (0.189 vs 0.045 for TV). This means each extra unit spent on Radio increases sales more than a unit spent on TV. The two results differ because TV spend has a much larger range than Radio spend, so coefficients are not directly comparable.
- Newspaper has almost no impact (importance 0.013, coefficient 0.003), so it appears to add very little to sales.

Based on this, TV and Radio are the channels that matter most for sales in this dataset.